In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder
import pickle

In [5]:
path="C:\MAIN\codenotes\AIML\project1\Customer-churn-prediction-ANN\dataset\Churn_Modelling.csv"

In [6]:
df=pd.read_csv(path)

In [7]:
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [8]:
#preprocessing

df=df.drop(['RowNumber','CustomerId','Surname'],axis=1)
df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [9]:
encodegender=LabelEncoder()

df['Gender']=encodegender.fit_transform(df['Gender'])
df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0


In [10]:
from sklearn.preprocessing import OneHotEncoder

encoder=OneHotEncoder()
encodedgeo=encoder.fit_transform(df[['Geography']])

encodedgeo


<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 10000 stored elements and shape (10000, 3)>

In [11]:
encoder.get_feature_names_out(['Geography'])

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [12]:
encoded_df=pd.DataFrame(encodedgeo.toarray(),columns=encoder.get_feature_names_out(['Geography']))

In [13]:
#combine encoded columns with original dataframe

df=pd.concat([df.drop('Geography',axis=1),encoded_df],axis=1)

In [14]:
df.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [15]:
#saving pkl file 

with open('encodegender.pkl', 'wb') as file:
    pickle.dump(encodegender, file)
    
with open('encoder.pkl','wb') as file:
    pickle.dump(encoder, file)

In [16]:
#dividing dataset

x=df.drop('Exited',axis=1)
y=df['Exited']

##splitting dataset

x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

#scalling
scaler=StandardScaler()
x_train=scaler.fit_transform(x_train)
x_test=scaler.transform(x_test)


In [17]:
with open('scaler.pkl','wb') as file:
    pickle.dump(scaler,file)

ANN Implementation

In [18]:


import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

In [19]:
x_train.shape[1]

12

In [20]:
#Building model

model=Sequential([
    Dense(64,activation='relu',input_shape=(x_train.shape[1],)),  ##1st hidden layer=input layer
    Dense(32,activation='relu'),     ##hiddenlayer2
    Dense(1,activation='sigmoid')
    
    
])

In [21]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                832       
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [22]:
opt=tf.keras.optimizers.Adam(learning_rate=0.01)
loss=tf.keras.losses.BinaryCrossentropy()

In [23]:
#model compilation

model.compile(optimizer=opt,loss=loss,metrics=['accuracy'])

In [24]:
#setup tensorboard

log_dir="logs/fit"+datetime.datetime.now().strftime("%Y%m%d%H%M%S")

In [25]:
tf_callback=TensorBoard(log_dir=log_dir,histogram_freq=1)

In [26]:
##setup for early stopping

earlystoppingcallback=EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True)



In [27]:
##TRAINING MODEL

history=model.fit(
    x_train,y_train,validation_data=(x_test,y_test),epochs=100,
    callbacks=[tf_callback,earlystoppingcallback]
    
)

Epoch 1/100


250/250 [==============================] - 2s 4ms/step - loss: 0.3960 - accuracy: 0.8361 - val_loss: 0.3609 - val_accuracy: 0.8570
Epoch 2/100
250/250 [==============================] - 0s 2ms/step - loss: 0.3557 - accuracy: 0.8540 - val_loss: 0.3801 - val_accuracy: 0.8470
Epoch 3/100
250/250 [==============================] - 0s 2ms/step - loss: 0.3467 - accuracy: 0.8581 - val_loss: 0.3760 - val_accuracy: 0.8440
Epoch 4/100
250/250 [==============================] - 0s 2ms/step - loss: 0.3470 - accuracy: 0.8602 - val_loss: 0.3409 - val_accuracy: 0.8570
Epoch 5/100
250/250 [==============================] - 0s 2ms/step - loss: 0.3386 - accuracy: 0.8615 - val_loss: 0.3590 - val_accuracy: 0.8525
Epoch 6/100
250/250 [==============================] - 0s 2ms/step - loss: 0.3397 - accuracy: 0.8619 - val_loss: 0.3446 - val_accuracy: 0.8590
Epoch 7/100
250/250 [==============================] - 0s 2ms/step - loss: 0.3347 - accuracy: 0.8639 - val_loss: 0.3393 - val_accuracy: 0.85

In [28]:
model.save('model.h5')

c:\MAIN\codenotes\AIML\project1\Customer-churn-prediction-ANN\venv\lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [29]:
##LOAD TENSORBOARD EXTENTSION

%reload_ext tensorboard

In [30]:
%tensorboard --logdir logs/fit20260414010609

ERROR: Failed to launch TensorBoard (exited with 1).
Contents of stderr:
Traceback (most recent call last):
  File "C:\MAIN\codenotes\AIML\project1\Customer-churn-prediction-ANN\venv\lib\runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "C:\MAIN\codenotes\AIML\project1\Customer-churn-prediction-ANN\venv\lib\runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "c:\MAIN\codenotes\AIML\project1\Customer-churn-prediction-ANN\venv\Scripts\tensorboard.exe\__main__.py", line 2, in <module>
  File "C:\MAIN\codenotes\AIML\project1\Customer-churn-prediction-ANN\venv\lib\site-packages\tensorboard\main.py", line 27, in <module>
    from tensorboard import default
  File "C:\MAIN\codenotes\AIML\project1\Customer-churn-prediction-ANN\venv\lib\site-packages\tensorboard\default.py", line 30, in <module>
    import pkg_resources
ModuleNotFoundError: No module named 'pkg_resources'